## Expected-result anchors (training split only)
Numbers behind `docs/expected_results.md`, written before any model is run on 2023 so the
expectations can't be fitted to the results. Only 2014-2022 is used.

In [1]:
import sys
from pathlib import Path
import numpy as np
# Read the src module
repo_root = Path.cwd().parent
sys.path.append(str(repo_root))
import yaml
from src.data.clean import clean_data
from src.data.split import get_split
with open(repo_root / "config" / "config.yaml") as f:
    config = yaml.safe_load(f)

### 1. Load the training split
`y` = `min_delay`, the target we predict (minutes).

In [2]:
train = get_split(clean_data(config), config, "training_set")
y = train["min_delay"]

Rows in 850590
dropped 39
Rows out 850551
mapped to Unknown: 65930
direction
East       192053
West       185255
North      173049
South      155067
Both        79197
Unknown     65930
Name: count, dtype: int64
Rows in 850551
dropped (missing / non-numeric / outside 1 - 999): 2849
Rows out 847702
distinct routes: 503
Rows in 847702
dropped total (target filter): 38020
Rows out 809682
training_set: 2014-2022, rows 683475


### 2. Median vs mean as a constant prediction
The naive benchmark predicts one number for every incident; any model must beat it.
MAE = average minutes off (primary metric). RMSE squares errors, so big misses count more (secondary).

In [3]:
median = y.median()
mean = y.mean()
print(f"median: {median:.2f}")
print(f"mean:   {mean:.2f}")
print(f"MAE  predicting median: {(y - median).abs().mean():.2f}")
print(f"MAE  predicting mean:   {(y - mean).abs().mean():.2f}")
print(f"RMSE predicting mean:   {np.sqrt(((y - mean) ** 2).mean()):.2f}")
print(f"RMSE predicting median: {np.sqrt(((y - median) ** 2).mean()):.2f}")

median: 10.00
mean:   13.42
MAE  predicting median: 7.17
MAE  predicting mean:   8.01
RMSE predicting mean:   15.06
RMSE predicting median: 15.45


- Median 10 is the benchmark; it is off by **7.17 min** on average, the bar models must beat.
- Median wins on MAE (7.17 vs 8.01), mean wins on RMSE (15.06 vs 15.45), so the median is the fair benchmark for an MAE task.
- Linear Regression minimises squared error, so expect it to do better on RMSE than on MAE.

### 3. Tail and mode
The tail = the few very long delays that pull the mean above the median.

In [4]:
print(f"share > 30 min: {(y > 30).mean():.3f}")
print(f"share > 60 min: {(y > 60).mean():.3f}")
for mode in ["bus", "streetcar"]:
    y_mode = y[train["mode"] == mode]
    print(f"{mode}: median {y_mode.median():.1f}, mean {y_mode.mean():.1f}, rows {len(y_mode)}")

share > 30 min: 0.040
share > 60 min: 0.016
bus: median 10.0, mean 13.8, rows 569214
streetcar: median 7.0, mean 11.7, rows 114261


- 4% of delays are over 30 min and 1.6% over 60 min: this tail is why mean > median.
- Streetcar delays are shorter than bus (median 7 vs 10), so mode, and probably route, should help predictions.

### 4. Trend by year
Checks whether 2023 is likely to differ from the training years (distribution shift).

In [5]:
year = train["timestamp"].dt.year
y_2022 = y[year == 2022]
print(f"2022 rows, MAE predicting the overall training median: {(y_2022 - median).abs().mean():.2f}")
train.groupby(year)["min_delay"].agg(["median", "mean", "count"]).round(2)

2022 rows, MAE predicting the overall training median: 7.70


,median,mean,count
timestamp,,,
2014,10.0,12.30,103341
2015,10.0,12.52,86767
2016,10.0,12.74,89290
2017,10.0,12.88,82118
2018,10.0,13.07,86849
2019,10.0,13.85,71634
2020,10.0,14.58,41378
2021,10.0,15.26,50908
2022,10.0,15.65,71190


- The median stays at 10, but the mean rises from 12.3 to 15.7: long delays are getting longer.
- 2022 alone gives MAE 7.70, so we expect the benchmark's 2023 MAE to be around 7.7-8.5, not 7.17.

### 5. Validation 2023 vs training (added Oct 19, after scoring)
**Not part of the pre-registration.** Sections 1–4 were written on Oct 12, before any model was
scored. This section was added on Oct 19, after the baseline results (Oct 13), to explain the
discrepancies in the M3 memo. It uses 2023 only; 2024 (test) is still untouched.

In [6]:
val = get_split(clean_data(config), config, "validation_set")
y_val = val["min_delay"]
for name, values in [("train 2014-2022", y), ("validation 2023", y_val)]:
    print(f"{name}: median {values.median():.1f}, mean {values.mean():.2f}, "
          f"share > 30 min {(values > 30).mean():.2%}, share > 60 min {(values > 60).mean():.2%}")

Rows in 850590
dropped 39
Rows out 850551
mapped to Unknown: 65930
direction
East       192053
West       185255
North      173049
South      155067
Both        79197
Unknown     65930
Name: count, dtype: int64
Rows in 850551
dropped (missing / non-numeric / outside 1 - 999): 2849
Rows out 847702
distinct routes: 503
Rows in 847702
dropped total (target filter): 38020
Rows out 809682
validation_set: 2023-2023, rows 62924
train 2014-2022: median 10.0, mean 13.42, share > 30 min 4.01%, share > 60 min 1.61%
validation 2023: median 11.0, mean 16.78, share > 30 min 6.06%, share > 60 min 2.68%


- 2023 has a heavier tail than training: median 10 → 11, mean 13.42 → 16.78, delays over 60 min
  1.61% → 2.68%. The mean was already rising every year (section 4); 2023 continued that trend.
- This is why the global median's RMSE (18.76) came out above our expected 16–18 while its MAE (8.28)
  stayed inside 7.7–8.5: RMSE squares errors, so more very long delays push it up far more than MAE.

### 6. Guessing the Decision Tree's score (added Nov 14, before the tree is trained)

**Goal:** write down how well we think the tree will do on 2023, *before* we train it.

**Problem:** we are not allowed to look at 2023 yet.

**Solution: a practice test inside the training years.**
- Learn from 2014–2021, then test on 2022. All of these years are training data, so this is allowed.
- We do **not** train a real tree here. We use a "fake tree" instead: put similar incidents into a group
  (for example, same route and same hour) and predict the group's average delay.
  A real tree does the same thing at each of its leaves.

In [31]:
from src.features.build import add_time_features

# print(train)
t_train = add_time_features(train)
year = t_train["timestamp"].dt.year
fit = t_train[year <= 2021]
mini_val  = t_train[year == 2022]
print(f"fit rows: {len(fit):,}   mini val rows: {len( mini_val ):,}   max year: {year.max()}")

fit rows: 612,285   mini val rows: 71,190   max year: 2022


In [32]:
from src.evaluate import mae, rmse

def group_lookup(cols, stat, min_n=1):
    """Predict each 2022 incident with the `stat` ("median" or "mean") of its group in 2014-2021.
    Groups with fewer than minimum number of training incidents (min_n) training rows fall back to route x hour,
    then to the global value."""
    groups = fit.groupby(cols)["min_delay"].agg([stat, "size"])
    # drops groups that are too small to trust
    groups = groups[groups["size"] >= min_n]
    pred = mini_val.join(groups[stat], on=cols)[stat]

    route_hour = fit.groupby(["route", "hour"])["min_delay"].agg(stat).rename("route_hour")
    pred = pred.fillna(mini_val.join(route_hour, on=["route", "hour"])["route_hour"])
    # print(pred)
    return pred.fillna(fit["min_delay"].agg(stat))


y_mini_val = mini_val["min_delay"]
prediction = group_lookup(["route", "hour"], "median")
print(f"route x hour median on 2022: MAE {mae(y_mini_val, prediction):.2f}")


route x hour median on 2022: MAE 6.39


In [33]:
import pandas as pd

lookups = [
    # (name for the table,                     columns to group by,               min_n)
    ("route x hour",                           ["route", "hour"],                 1),
    ("route x hour x incident",                ["route", "hour", "incident"],     1),
    ("route x hour x incident, min_n=20",      ["route", "hour", "incident"],     20),
    ("route x incident x direction, min_n=20", ["route", "incident", "direction"], 20),
    ("mode x incident x hour",                 ["mode", "incident", "hour"],      1),
]

rows = []
for stat in ["median", "mean"]:
    # Global: predict one number (the training median or mean) for every incident
    pred = pd.Series(fit["min_delay"].agg(stat), index=mini_val.index)
    rows.append({"grouping": "global (one number)", "stat": stat,
                 "MAE": mae(y_mini_val, pred), "RMSE": rmse(y_mini_val, pred)})
print(rows[0])

for name, cols, min_n in lookups:
    for stat in ["median", "mean"]:
        pred = group_lookup(cols, stat, min_n)
        rows.append({"grouping": name, "stat": stat,
                     "MAE": mae(y_mini_val, pred), "RMSE": rmse(y_mini_val, pred)})

print(rows[10])

table = pd.DataFrame(rows).pivot(index="grouping", columns="stat", values=["MAE", "RMSE"]).round(2)
#  It shows how much worse predicting the mean is, i.e. what a tree leaf costs us.
table["mean penalty (MAE)"] = table[("MAE", "mean")] - table[("MAE", "median")]
table.sort_values(("MAE", "median"), ascending=False)

{'grouping': 'global (one number)', 'stat': 'median', 'MAE': 7.698777918246945, 'RMSE': 17.731820954554472}
{'grouping': 'mode x incident x hour', 'stat': 'median', 'MAE': 7.320571709509762, 'RMSE': 15.34263045154372}


MAE          RMSE         \
stat                                    mean median   mean median   
grouping                                                            
global (one number)                     8.16   7.70  16.99  17.73   
mode x incident x hour                  7.31   7.32  13.95  15.34   
route x hour                            6.91   6.39  16.23  16.79   
route x hour x incident, min_n=20       6.58   6.30  15.99  16.63   
route x hour x incident                 6.36   6.21  14.93  15.71   
route x incident x direction, min_n=20  6.41   6.21  15.10  16.00   

                                       mean penalty (MAE)  
stat                                                       
grouping                                                   
global (one number)                                  0.46  
mode x incident x hour                              -0.01  
route x hour                                         0.52  
route x hour x incident, min_n=20                    0.28  
route x hour x incident                              0.15  
route x incident x direction, min_n=20               0.20


- A tree predicts the group's **average (mean)**, so we look at the
  "MAE mean" column.
- The best groupings that include route score **6.36 to 6.58**.
  Our best model so far, the route × hour **median**, scores **6.39**. So they are about the same.
- Two effects cancel out:
  - Smaller, more specific groups help a little: about **0.2 min** better.
  - Predicting the average instead of the middle value (median) hurts: **0.15 to 0.5 min** worse
    (the "mean penalty" column). A few very long delays pull the average up, so it is too high
    for most normal incidents.
-  RMSE punishes big misses heavily. Predicting the average is better for big misses,
  so RMSE drops a lot: **13.95 to 15.10**, compared with **16.79** for the route × hour median.

**From the 2022 practice test to a 2023 guess**

2023 has longer delays than 2022:
the median goes from 10 to **11** min and the mean from 15.65 to **16.78** min
(2022 from the Section 4 table, 2023 from the Section 5 output).
We can also measure what that does to the scores, because the baselines were scored on both years
(2022 in the table above, 2023 in `results/baseline_table.csv` from M3):

| Baseline | 2022 | 2023 | How much worse in 2023 |
|---|---|---|---|
| Global median, MAE | 7.70 | 8.28 | +0.58 |
| Route × hour median, MAE | 6.39 | 6.97 | +0.58 |
| Global mean, MAE | 8.16 | 8.55 | +0.39 |
| RMSE (all three baselines) | | | about 5% higher |

So a model gets about **0.4 to 0.6 min worse** on MAE in 2023, and about **5% worse** on RMSE.

**Guess the tree's MAE.**
- Fake tree on 2022: 6.36 to 6.58.
- Add the 2023 difference: 6.36 + 0.4 = **6.76** (best case) up to 6.58 + 0.6 = **7.18** (worst case).
- A real tree chooses its own groups, so it may do a little better than our hand-made ones.
  We lower the bottom of the range a little, to 6.6.
- **Expected tree MAE on 2023: 6.6 to 7.2.**

**Guess the tree's RMSE.**
- Fake tree on 2022: 13.95 to 15.10. Add 5%: **14.6 to 15.9**.
- A real tree is built to reduce big misses (squared error), so it may do slightly better.
- **Expected tree RMSE on 2023: 14.0 to 16.0.**

On MAE the tree will be **about the same** as the route × hour median (6.97).
On RMSE it will be **clearly better** than the route × hour median (17.62), and close to Linear Regression (14.62).